In [0]:
import os
from pathlib import Path
V3   = "mlo.features.weather_daily_v3"
V3_VERSION = spark.sql(f"DESCRIBE HISTORY {V3}").selectExpr("max(version) AS v").first()["v"]

def git_commit(start=None):
    """Best effort — returns 'unknown' rather than breaking a training run."""
    try:
        root = Path(start or os.getcwd())
        for d in (root, *root.parents):
            head = d / ".git" / "HEAD"
            if head.exists():
                ref = head.read_text().strip()
                if ref.startswith("ref:"):
                    return (d / ".git" / ref.split(" ", 1)[1]).read_text().strip()
                return ref
    except Exception:
        pass
    return "unknown"

GIT_COMMIT = git_commit()

print(f"v3 version {V3_VERSION} | commit {GIT_COMMIT}")

In [0]:
import mlflow
import pandas as pd
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

fe = FeatureEngineeringClient()

# Source features through the Feature Store rather than the flat snapshot. This is what
# lets fe.log_model() record which feature table and lookup keys produced the model --
# lineage that survives into Unity Catalog and makes the model servable with automatic
# lookups. Labels drive the join, same as 03.
lookups = [FeatureLookup(table_name=V3, lookup_key=["station"], timestamp_lookup_key="date")]
ts = fe.create_training_set(df=spark.table("mlo.features.weather_labels"),
                            feature_lookups=lookups, label="Bad")

pdf = ts.load_df().toPandas().sort_values("date").reset_index(drop=True)

FEATURES = [c for c in pdf.columns if c not in ("station", "date", "Bad")]
split = int(len(pdf) * 0.8)                     # chronological, matching 03
train, test = pdf.iloc[:split], pdf.iloc[split:]
Xtr, ytr = train[FEATURES], train["Bad"]
Xte, yte = test[FEATURES], test["Bad"]

print(f"{len(FEATURES)} features | "
      f"train {len(train)} ({train.date.min().date()} -> {train.date.max().date()}) | "
      f"test {len(test)} ({test.date.min().date()} -> {test.date.max().date()})")

mlflow.set_experiment("/Shared/adsp32021_weather")

CANDIDATES = {
    "logreg": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                   solver="liblinear", random_state=42))]),
    # L1 does the feature selection that 44 features against ~475 events demands.
    "logreg_l1": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", penalty="l1",
                                   C=0.1, solver="liblinear", random_state=42))]),
    "random_forest": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                       min_samples_leaf=5, random_state=42, n_jobs=-1))]),
    # Handles NaN natively -- no imputer, which is the honest treatment for a table
    # where a null means "that station didn't report", not "unknown value".
    "hist_gbm": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                               random_state=42),
}

results = []
for name, model in CANDIDATES.items():
    with mlflow.start_run(run_name=f"{name}_v3"):
        mlflow.set_tags({"feature_set_version": "v3", "model_type": name,
                         "target": "Bad_t+1", "source": "manual_sweep",
                         "git_commit": GIT_COMMIT})
        mlflow.log_input(mlflow.data.load_delta(table_name=V3, version=V3_VERSION),
                         context="features")
        mlflow.log_params({"feature_delta_version": V3_VERSION, "n_features": len(FEATURES)})
        mlflow.log_dict({"feature_set": "v3",
                         "columns": FEATURES,
                         "defined_by": "data_pipelines/11_features_nightly.ipynb",
                         "source_table": V3,
                         "source_version": V3_VERSION,
                         "git_commit": GIT_COMMIT},
                        "feature_manifest.json")

        model.fit(Xtr, ytr)
        pred, proba = model.predict(Xte), model.predict_proba(Xte)[:, 1]
        m = {"f1": f1_score(yte, pred, zero_division=0),
             "roc_auc": roc_auc_score(yte, proba),
             "accuracy": accuracy_score(yte, pred),
             "precision": precision_score(yte, pred, zero_division=0),
             "recall": recall_score(yte, pred, zero_division=0)}
        mlflow.log_metrics(m)

        # fe.log_model, not mlflow.sklearn.log_model: records the feature table and lookup
        # keys on the model, so a served endpoint resolves features itself instead of the
        # caller supplying all 44 columns. Also infers the signature from input_example,
        # which is what UC registration was rejecting.
        # No registered_model_name on purpose -- four sweep models per run would clutter
        # the registry. Register the winner through 03.
        fe.log_model(model=model, artifact_path=name, flavor=mlflow.sklearn,
                     training_set=ts, input_example=Xtr.head(3))

        results.append({"model": name, **m})

print(pd.DataFrame(results).sort_values("f1", ascending=False).to_string(index=False))

In [0]:
from flaml import AutoML
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Need to impute on train.
imp = SimpleImputer(strategy="median").fit(Xtr)
Xtr_f, Xte_f = imp.transform(Xtr), imp.transform(Xte)

fa = AutoML()
with mlflow.start_run(run_name="flaml_v3"):
    mlflow.set_tags({"feature_set_version": "v3", "model_type": "flaml",
                     "target": "Bad_t+1", "source": "flaml", "git_commit": GIT_COMMIT})
    mlflow.log_input(mlflow.data.load_delta(table_name=V3, version=V3_VERSION), context="features")
    mlflow.log_param("feature_delta_version", V3_VERSION)
    mlflow.log_dict({"feature_set": "v3", "columns": FEATURES,
                     "defined_by": "data_pipelines/11_features_nightly.ipynb",
                     "source_table": V3, "source_version": V3_VERSION,
                     "git_commit": GIT_COMMIT}, "feature_manifest.json")

    fa.fit(X_train=Xtr_f, y_train=ytr, task="classification", metric="f1",
           time_budget=300, mlflow_logging=False, verbose=1)

    pred, proba = fa.predict(Xte_f), fa.predict_proba(Xte_f)[:, 1]
    mlflow.log_params({"best_estimator": fa.best_estimator, "n_features": len(FEATURES)})
    mlflow.log_metrics({"f1": f1_score(yte, pred, zero_division=0),
                        "roc_auc": roc_auc_score(yte, proba),
                        "accuracy": accuracy_score(yte, pred),
                        "precision": precision_score(yte, pred, zero_division=0),
                        "recall": recall_score(yte, pred, zero_division=0)})

    # Bundle the fitted imputer with the fitted estimator
    flaml_model = Pipeline([("impute", imp), ("clf", fa.model.estimator)])
    fe.log_model(model=flaml_model, artifact_path="flaml_best", flavor=mlflow.sklearn,
                 training_set=ts, input_example=Xtr.head(3))

print(fa.best_estimator, fa.best_config)